# 빠알리어 TTS 생성기 (Indic Parler TTS 산스크리트어)

SuttaLog2 앱용 고품질 빠알리어 음성 파일 생성

**사용법**: 런타임 → 모두 실행

In [ ]:
# 1단계: GPU 확인 + 패키지 설치
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q parler-tts soundfile numpy pydub
!apt-get -qq install ffmpeg
print('설치 완료!')

In [ ]:
# 2단계: 모델 로드
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf
import os, re, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model_name = 'ai4bharat/indic-parler-tts'
model = ParlerTTSForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print('모델 로드 완료!')

In [ ]:
# 3단계: tts-texts.json 업로드
from google.colab import files
uploaded = files.upload()
texts = json.loads(list(uploaded.values())[0].decode('utf-8'))
print(f'{len(texts)}개 텍스트 로드 완료')

In [ ]:
# 4단계: 음성 생성
os.makedirs('audio', exist_ok=True)

def text_to_filename(text):
    name = text.lower().strip()
    name = re.sub(r'[\n\r]', ' ', name)
    name = re.sub(r'[^a-z\u0101\u012b\u016b\u1e6d\u1e0d\u1e47\u1e45\u00f1\u1e43\u1e37 0-9\s\-]', '', name)
    name = re.sub(r'\s+', '-', name.strip())
    return name[:80]

description = 'A male speaker speaks clearly and slowly in Sanskrit with a calm and steady tone.'
desc_ids = tokenizer(description, return_tensors='pt').input_ids.to(device)

total = len(texts)
errors = []

for i, text in enumerate(texts):
    fname = text_to_filename(text)
    outpath = f'audio/{fname}.wav'
    if os.path.exists(outpath):
        print(f'[{i+1}/{total}] skip: {fname}')
        continue
    try:
        clean_text = text.replace('\n', ' ').strip()
        input_ids = tokenizer(clean_text, return_tensors='pt').input_ids.to(device)
        with torch.no_grad():
            generation = model.generate(
                input_ids=input_ids,
                prompt_input_ids=desc_ids,
                max_new_tokens=500,
            )
        audio = generation.cpu().numpy().squeeze()
        sf.write(outpath, audio, samplerate=22050)
        print(f'[{i+1}/{total}] {fname} ({len(audio)/22050:.1f}s)')
    except Exception as e:
        errors.append((text, str(e)))
        print(f'[{i+1}/{total}] ERR: {fname} - {e}')

print(f'\n완료! 성공: {total - len(errors)}, 오류: {len(errors)}')

In [ ]:
# 5단계: MP3 변환 + ZIP 다운로드
from pydub import AudioSegment
import glob, zipfile

os.makedirs('audio_mp3', exist_ok=True)
for wf in glob.glob('audio/*.wav'):
    mp3f = wf.replace('audio/', 'audio_mp3/').replace('.wav', '.mp3')
    if not os.path.exists(mp3f):
        AudioSegment.from_wav(wf).export(mp3f, format='mp3', bitrate='64k')

mapping = {}
for text in texts:
    fname = text_to_filename(text)
    if os.path.exists(f'audio_mp3/{fname}.mp3'):
        mapping[text] = f'{fname}.mp3'

with open('audio_mp3/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile('pali-audio.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk('audio_mp3'):
        for fn in fnames:
            zf.write(os.path.join(root, fn), fn)

size_mb = os.path.getsize('pali-audio.zip') / (1024*1024)
print(f'pali-audio.zip ({size_mb:.1f} MB) - {len(mapping)}개 파일')
files.download('pali-audio.zip')